## Tensorflow를 활용한 딥러닝 3

#### 1. File Search

In [2]:
import requests
from io import BytesIO
from openai import OpenAI

client = OpenAI()

def create_file(client, file_path):
    if file_path.startswith("http://") or file_path.startswith("https://"):
        # Download the file content from the URL
        response = requests.get(file_path)
        file_content = BytesIO(response.content)
        file_name = file_path.split("/")[-1]
        file_tuple = (file_name, file_content)
        result = client.files.create(
            file=file_tuple,
            purpose="assistants"
        )
    else:
        # Handle local file path
        with open(file_path, "rb") as file_content:
            result = client.files.create(
                file=file_content,
                purpose="assistants"
            )
    print(result.id)
    return result.id

# Replace with your own file path or URL
file_id = create_file(client, "howto-sockets.pdf")


file-71ViaoaUZ5hoizG75FNKrp


In [3]:
vector_store = client.vector_stores.create(
    name='knowledge_base'
)

print(vector_store.id)

vs_685cb692d75c8191acaceae5e83d6775


In [5]:
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_id
)

VectorStoreFile(id='file-71ViaoaUZ5hoizG75FNKrp', created_at=1750906671, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_685cb692d75c8191acaceae5e83d6775', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [7]:
result_list = client.vector_stores.files.list(
    vector_store_id=vector_store.id
)

print(result_list)

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-71ViaoaUZ5hoizG75FNKrp', created_at=1750906671, last_error=None, object='vector_store.file', status='completed', usage_bytes=29112, vector_store_id='vs_685cb692d75c8191acaceae5e83d6775', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-71ViaoaUZ5hoizG75FNKrp', last_id='file-71ViaoaUZ5hoizG75FNKrp')


In [12]:
response = client.responses.create(
    model='gpt-4.1',
    input='파이썬 코드로 소켓 만드는 방법을 간단하게 설명해줘',
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    tool_choice='required'
)

print(response)

Response(id='resp_685cd098cf28819a894b99bb5c27b9a304db3f3932f8abfc', created_at=1750913176.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseFileSearchToolCall(id='fs_685cd099419c819aa67ec29ee96f71db04db3f3932f8abfc', queries=['파이썬 소켓 만드는 방법', '파이썬 소켓 예제 코드', '파이썬에서 소켓을 생성하는 방법을 설명해줘'], status='completed', type='file_search_call', results=None), ResponseOutputMessage(id='msg_685cd09b7080819a8e578a4da367b0c504db3f3932f8abfc', content=[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-71ViaoaUZ5hoizG75FNKrp', filename='howto-sockets.pdf', index=1189, type='file_citation'), AnnotationFileCitation(file_id='file-71ViaoaUZ5hoizG75FNKrp', filename='howto-sockets.pdf', index=1189, type='file_citation')], text="파이썬에서 소켓을 만드는 방법을 간단하게 설명하면 다음과 같습니다.\n\n---\n\n### 1. 소켓 생성\n\n```python\nimport socket\ns = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n```\n- `AF_INET`: IPv4용 소켓을 생성.\n- `SOC

In [11]:
print(dict(response))

{'id': 'resp_685cd07172788199992de35967cca271066d5d63d73bb639', 'created_at': 1750913137.0, 'error': None, 'incomplete_details': None, 'instructions': None, 'metadata': {}, 'model': 'gpt-4.1-2025-04-14', 'object': 'response', 'output': [ResponseOutputMessage(id='msg_685cd07205b081998cee4bb52041bacc066d5d63d73bb639', content=[ResponseOutputText(annotations=[], text="파이썬에서 소켓을 만드는 방법을 간단하게 설명드리겠습니다.\n\n1. **socket 라이브러리 임포트**\n   ```python\n   import socket\n   ```\n\n2. **소켓 객체 생성**\n   - IPv4와 TCP 소켓을 만들 때:\n     ```python\n     s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n     ```\n\n3. **서버 소켓 예시**\n   ```python\n   import socket\n\n   # 소켓 생성\n   s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n   # IP와 포트 바인딩\n   s.bind(('localhost', 12345))\n   # 연결 대기\n   s.listen()\n\n   print('서버 대기 중...')\n   conn, addr = s.accept()\n   print('연결:', addr)\n\n   # 데이터 주고받기\n   data = conn.recv(1024)\n   print('받은 데이터:', data.decode())\n\n   conn.sendall('안녕하세요!'.encode())\n   co

### 2. Streamlit
- `streamlit run <파일명.py>`으로 실행

In [ ]:
import streamlit as st

st.set_page_config(
    page_title="Hello"
)

st.write("# Welcome to Streamlit!")

st.sidebar.success("데모 선택.")

st.markdown(
    """
    Streamlit은 머신 러닝 및 데이터 과학 프로젝트를 위해 특별히 제작된 오픈 소스 앱 프레임워크입니다..
    ### 자세히 알아보고 싶으신가요?
    -streamlit.io](https://streamlit.io) 
    -[설명서](https://docs.streamlit.io) 
    -[커뮤니티 포럼](https://discuss.streamlit.io)에서 질문하기 
    """
)

In [ ]:
import streamlit as st
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

st.title('ChatGPT')

# st.session_state : 세션에 키-값 형식으로 데이터를 저장하는 변수
# openai_model: str, message: []
if 'openai_model' not in st.session_state:
    st.session_state.openai_model = 'gpt-4.1'   # 'gpt-3.5-turbo'

if 'messages' not in st.session_state:
    st.session_state.messages = []

# 기존의 메시지가 있다면 출력
for msg in st.session_state.messages:
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])

# prompt: 사용자 입력창
if prompt:= st.chat_input('메시지를 입력하세요!') :
    # st.write(prompt)
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message('user'):
        st.markdown(prompt)

    with st.chat_message('assistant'):
        stream = client.chat.completions.create(
            model=st.session_state.openai_model,
            messages=[
                {"role": m['role'], "content": m['content']}
                for m in st.session_state.messages
            ],
            stream=True
        )
        response = st.write_stream(stream)
    
    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": response
        }
    )

#### OpenAi API와 Streamlit를 활용한 영어 회화 튜터 챗봇 만들기

In [ ]:
import streamlit as st
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

st.set_page_config(page_title="영어 회화 튜터", page_icon="🤠")
st.title('영어 회화 튜터 🤠')
st.markdown(
    """
    이 챗봇은 영어 회화 연습을 위한 AI 튜터입니다. 영어로 자유롭게 질문하거나 대화를 시작해보세요!  
    AI가 영어로 답변을 제공하고, 괄호 안에 한글 번역도 함께 안내해줍니다.  
    또한, 문법 교정이나 자연스러운 표현, 영어 학습에 도움이 되는 피드백도 받을 수 있습니다.

    **이런 분께 추천합니다**  
    - 영어로 실전 대화를 연습하고 싶은 분  
    - 자신의 영어 문장을 교정받고 싶은 분  
    - 영어 표현력과 이해력을 함께 키우고 싶은 분

    아래 입력창에 영어 또는 한글로 질문을 입력해보세요.  
    AI와 함께 쉽고 재미있게 영어 실력을 향상시켜보세요!
    """
)

with st.expander("💡 대화 예시 보기"):
    st.markdown(
        """
            **User:** Hello! How are you?  
            **Assistant:** I'm great, thank you! How about you? (저는 잘 지내고 있어요, 감사합니다! 당신은요?)

            **User:** Can you help me practice English?  
            **Assistant:** Of course! Let's start a conversation. (물론이죠! 대화를 시작해봅시다.)

            **User:** What did you do yesterday?  
            **Assistant:** I spent my day reading books and learning new things. (저는 어제 책을 읽고 새로운 것을 배우며 보냈어요.)
        """
    )

st.divider()

# st.session_state : 세션에 키-값 형식으로 데이터를 저장하는 변수
# openai_model: str, message: []
if 'openai_model' not in st.session_state:
    st.session_state.openai_model = 'gpt-4.1'

if 'messages' not in st.session_state:
    st.session_state.messages = [{
        "role": "system",
        "content": "당신은 영어 회화 튜터입니다. 영어로만 대화하고, 밑에 괄호로 한글로 번역된 내용을 작성하는 형식으로 대화해주세요. 문법 교정이나 실전 회화에 도움이 될만한 영어 학습 피드백도 제공해주세요"
    }]

# 이전 대화 출력
for msg in st.session_state.messages:
    if msg['role'] == 'system':
        continue
    with st.chat_message(msg['role']):
        st.markdown(msg['content'])

# prompt: 사용자 입력창
if prompt:= st.chat_input('메시지를 입력하세요!') :
    # st.write(prompt)
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message('user'):
        st.markdown(prompt)

    with st.chat_message('assistant'):
        stream = client.chat.completions.create(
            model=st.session_state.openai_model,
            messages=[
                {"role": m['role'], "content": m['content']}
                for m in st.session_state.messages
            ],
            stream=True
        )
        response = st.write_stream(stream)
    
    st.session_state.messages.append(
        {
            "role": "assistant",
            "content": response
        }
    )

# 사용자가 입력한 메시지가 있을 때만 버튼 표시
if any(msg['role'] == 'user' for msg in st.session_state.messages):
    # 대화 내용 텍스트 파일로 저장
    if st.button("💾 대화 내용 저장하기"):
        # system 메시지는 제외하고 저장
        chat_log = ""
        for msg in st.session_state.messages:
            if msg['role'] == 'system':
                continue
            role = "user" if msg['role'] == 'user' else "assistant"
            chat_log += f"{role}: {msg['content']}\n\n"
        # 텍스트 파일로 저장
        with open('chat_log.txt', 'w', encoding='utf-8') as f:
            f.write(chat_log)
        st.success("대화 내용이 chat_log.txt 파일로 저장되었습니다")